# GFP fluorescence prediction

ESM-2 35M as the base model, ProtGPT2 as a second model from a different family (autoregressive
instead of masked). Both are frozen, only the ridge regression head and a small mapping network
get trained.

In [ ]:
!pip install -q transformers tokenizers scikit-learn scipy pandas

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import os

torch.manual_seed(0)
np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
# load the split, files can be loose or inside a dataset folder
def load(name):
    for p in [name, "dataset/" + name]:
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(name)

train = load("train.csv")
val = load("validation.csv")
test = load("test.csv")
print(len(train), len(val), len(test))

In [ ]:
# sanity check the data before doing anything else
print(train[["mutant", "DMS_score"]].head())
print(set(train["mutated_sequence"].str.len()))          # should all be 238
print((pd.concat([train, val, test])["mutation_count"] == 1).all())   # single mutants only

## get embeddings from both models

one vector per sequence, mean pooled over the residues from the last layer

In [ ]:
from transformers import AutoTokenizer, AutoModel

esm_tok = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
esm_model = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D").to(device).eval()

def esm_embed(seqs, batch_size=32):
    feats = []
    for i in range(0, len(seqs), batch_size):
        batch = seqs[i:i+batch_size]
        enc = esm_tok(batch, return_tensors="pt", padding=True)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            h = esm_model(**enc).last_hidden_state
        # esm adds a cls token at the start and an eos token at the end, drop both before pooling
        mask = enc["attention_mask"].clone()
        mask[:, 0] = 0
        lengths = enc["attention_mask"].sum(1)
        for j, L in enumerate(lengths):
            mask[j, L-1] = 0
        mask = mask.unsqueeze(-1).float()
        pooled = (h * mask).sum(1) / mask.sum(1)
        feats.append(pooled.cpu().numpy())
    return np.concatenate(feats)

In [ ]:
# tried ProGen2-small first but its custom model code kept breaking (get_head_mask missing,
# then tensors stuck on a meta device) with the transformers version colab installs.
# switched to ProtGPT2 instead, still autoregressive and a different family from esm,
# and it's just a plain gpt2 model so it loads without any of that trouble.

from transformers import AutoModelForCausalLM

pg_tok = AutoTokenizer.from_pretrained("nferruz/ProtGPT2")
pg_tok.pad_token = pg_tok.eos_token
pg_model = AutoModelForCausalLM.from_pretrained("nferruz/ProtGPT2").to(device).eval()

def pg_embed(seqs, batch_size=16):
    feats = []
    for i in range(0, len(seqs), batch_size):
        batch = seqs[i:i+batch_size]
        enc = pg_tok(batch, return_tensors="pt", padding=True)
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            h = pg_model(**enc, output_hidden_states=True).hidden_states[-1]
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (h * mask).sum(1) / mask.sum(1)
        feats.append(pooled.float().cpu().numpy())
    return np.concatenate(feats)

In [ ]:
# this takes a while so cache the embeddings to disk, reruns just load the npy files
def get_feats(name, seqs, fn):
    path = name + ".npy"
    if os.path.exists(path):
        return np.load(path)
    f = fn(list(seqs))
    np.save(path, f)
    return f

esm_train = get_feats("esm_train", train["mutated_sequence"], esm_embed)
esm_val = get_feats("esm_val", val["mutated_sequence"], esm_embed)
esm_test = get_feats("esm_test", test["mutated_sequence"], esm_embed)

pg_train = get_feats("pg_train", train["mutated_sequence"], pg_embed)
pg_val = get_feats("pg_val", val["mutated_sequence"], pg_embed)
pg_test = get_feats("pg_test", test["mutated_sequence"], pg_embed)

y_train = train["DMS_score"].values
y_val = val["DMS_score"].values
y_test = test["DMS_score"].values

print(esm_train.shape, pg_train.shape)

## prediction head and metrics

ridge regression everywhere so the comparison is about the features, not the head

In [ ]:
def evaluate(pred, true):
    rho = spearmanr(pred, true).correlation
    mse = float(((pred - true) ** 2).mean())
    return rho, mse

# just grid search alpha and keep whatever gets the best validation spearman
def pick_alpha(Xtr, ytr, Xval, yval):
    best = None
    for a in [0.1, 1, 10, 100, 1000]:
        m = Ridge(alpha=a).fit(Xtr, ytr)
        rho, mse = evaluate(m.predict(Xval), yval)
        if best is None or rho > best[1]:
            best = (a, rho, mse)
    return best

## baselines

fit on train (657), pick alpha on validation (214)

In [ ]:
esm_scaler = StandardScaler().fit(esm_train)
pg_scaler = StandardScaler().fit(pg_train)

esm_tr = esm_scaler.transform(esm_train)
esm_va = esm_scaler.transform(esm_val)
pg_tr = pg_scaler.transform(pg_train)
pg_va = pg_scaler.transform(pg_val)

print("esm only    ", pick_alpha(esm_tr, y_train, esm_va, y_val))
print("protgpt2    ", pick_alpha(pg_tr, y_train, pg_va, y_val))

concat_tr = np.concatenate([esm_tr, pg_tr], axis=1)
concat_va = np.concatenate([esm_va, pg_va], axis=1)
print("concat both ", pick_alpha(concat_tr, y_train, concat_va, y_val))

## transfer method

based on the SoupFold paper's idea of mapping one model's representation into another's space.
here I train a small MLP to predict the esm vector from the protgpt2 vector (mse loss), then
average the real esm vector with the mapped protgpt2 vector and run ridge on that.

In [ ]:
class Mapper(nn.Module):
    def __init__(self, din, dout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(din, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, dout),
        )
    def forward(self, x):
        return self.net(x)

def train_mapper(src, tgt, epochs=300):
    src = torch.tensor(src, dtype=torch.float32, device=device)
    tgt = torch.tensor(tgt, dtype=torch.float32, device=device)
    model = Mapper(src.shape[1], tgt.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=3e-4)
    loss_fn = nn.MSELoss()
    for epoch in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(src), tgt)
        loss.backward()
        opt.step()
    return model

def map_feats(model, src):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(src, dtype=torch.float32, device=device)).cpu().numpy()

In [ ]:
mapper = train_mapper(pg_tr, esm_tr)

# check how well the mapper actually reconstructs esm features (not the real metric, just a check)
pred_va = map_feats(mapper, pg_va)
ss_res = ((esm_va - pred_va) ** 2).sum()
ss_tot = ((esm_va - esm_va.mean(0)) ** 2).sum()
print("mapper r2 on val:", 1 - ss_res / ss_tot)

comb_tr = (esm_tr + map_feats(mapper, pg_tr)) / 2
comb_va = (esm_va + pred_va) / 2
print("transfer    ", pick_alpha(comb_tr, y_train, comb_va, y_val))

## final refit and test

settings picked from the validation numbers above. refit everything on train+val (871) and
score once on the 213 test variants.

In [ ]:
# alphas chosen from the validation results printed above
alpha_esm = 10
alpha_concat = 1
alpha_transfer = 0.1

esm_dev = np.concatenate([esm_train, esm_val])
pg_dev = np.concatenate([pg_train, pg_val])
y_dev = np.concatenate([y_train, y_val])

esm_scaler = StandardScaler().fit(esm_dev)
pg_scaler = StandardScaler().fit(pg_dev)
esm_d = esm_scaler.transform(esm_dev)
esm_t = esm_scaler.transform(esm_test)
pg_d = pg_scaler.transform(pg_dev)
pg_t = pg_scaler.transform(pg_test)

esm_pred = Ridge(alpha=alpha_esm).fit(esm_d, y_dev).predict(esm_t)
print("esm only test:", evaluate(esm_pred, y_test))

concat_pred = Ridge(alpha=alpha_concat).fit(
    np.concatenate([esm_d, pg_d], axis=1), y_dev).predict(np.concatenate([esm_t, pg_t], axis=1))
print("concat test:  ", evaluate(concat_pred, y_test))

mapper = train_mapper(pg_d, esm_d)
comb_d = (esm_d + map_feats(mapper, pg_d)) / 2
comb_t = (esm_t + map_feats(mapper, pg_t)) / 2
transfer_pred = Ridge(alpha=alpha_transfer).fit(comb_d, y_dev).predict(comb_t)
print("transfer test:", evaluate(transfer_pred, y_test))

In [ ]:
# save one prediction file per method, matching the submission template columns
for name, pred in [("esm", esm_pred), ("concat", concat_pred), ("transfer", transfer_pred)]:
    pd.DataFrame({"variant_id": test["variant_id"], "predicted_fitness": pred}).to_csv(
        f"pred_{name}.csv", index=False)
print("done")